# Evaluate Trained Checkpoint

Load a ScipherModel checkpoint and run validation on the held-out split.

**Runs on cluster** (needs SOMA database + ESM-2 embeddings).

## 1. Setup & Config

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt

import tiledbsoma as soma
from tiledbsoma_ml import ExperimentDataset, experiment_dataloader

# Find project root by searching upward for pyproject.toml
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "pyproject.toml").exists():
        PROJECT_ROOT = _parent
        break
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")

from scipher.hierarchy import (
    load_prebuilt_hierarchy, MarginalizationLoss, WideNN,
)
from scipher.embedders.attention import AttentionPooling

In [ ]:
# --- Config ---
# Point to the checkpoint you want to evaluate
CHECKPOINT_PATH = DATA_DIR / "checkpoint" / "2026-03-11_2026-01-29_CL0000988" / "epoch10.pt"

HIERARCHY_DATE = "2026-01-29"
ROOT_CL_ID = "CL:0000988"
SOMA_URI = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
MIN_CELL_COUNT = 50
BATCH_SIZE = 64
SEED = 42
NUM_WORKERS = 6

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
effective_batch_size = BATCH_SIZE * max(num_gpus, 1)

# List available checkpoints for reference
checkpoint_base = DATA_DIR / "checkpoint"
if checkpoint_base.exists():
    for run_dir in sorted(checkpoint_base.iterdir()):
        if run_dir.is_dir():
            epochs = sorted(run_dir.glob("*.pt"))
            print(f"  {run_dir.name}/  ({len(epochs)} checkpoints)")
            for ep in epochs:
                print(f"    {ep.name}")

print(f"\nLoading: {CHECKPOINT_PATH}")
print(f"Device: {device}")
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

## 2. Load Checkpoint & Training Loss Curves

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

print(f"Checkpoint epoch: {ckpt['epoch']}")
print(f"Config: {ckpt['config']}")
if "epoch_times" in ckpt:
    print(f"Epoch times: {['%.1fs' % t for t in ckpt['epoch_times']]}")
    print(f"Total training time: {sum(ckpt['epoch_times']):.1f}s")

In [ ]:
loss_history = ckpt.get("loss_history", {})

def smooth(values, window=50):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

if loss_history:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, key, title in zip(
        axes, ["total", "leaf", "parent"],
        ["Total Loss", "Leaf Loss", "Parent Loss"],
    ):
        vals = loss_history.get(key, [])
        if not vals:
            ax.set_title(f"{title} (no data)")
            continue
        if len(vals) > 50:
            ax.plot(smooth(vals), alpha=0.8, label="smoothed")
            ax.plot(vals, alpha=0.15, color="gray", label="raw")
            ax.legend()
        else:
            ax.plot(vals, alpha=0.8)
        ax.set_title(title)
        ax.set_xlabel("Batch")
        ax.set_ylabel("Loss")

    plt.suptitle(f"Training Loss (checkpoint epoch {ckpt['epoch']})", y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"Final total loss: {loss_history['total'][-1]:.4f}")
    print(f"Final leaf loss:  {loss_history['leaf'][-1]:.4f}")
    print(f"Final parent loss: {loss_history['parent'][-1]:.4f}")
else:
    print("No loss_history in checkpoint")

## 3. Data Loading

In [ ]:
(
    mapping_dict, leaf_values, internal_values,
    marginalization_df, parent_child_df, exclusion_df,
) = load_prebuilt_hierarchy(HIERARCHY_DATE, ROOT_CL_ID)

all_cell_values = list(leaf_values) + list(internal_values)
idx_to_cl = {v: k for k, v in mapping_dict.items()}
leaf_idx_set = {mapping_dict[cl] for cl in leaf_values}
print(f"Leaf types: {len(leaf_values)}, Internal: {len(internal_values)}, Total: {len(all_cell_values)}")

In [ ]:
EMB_PATH = DATA_DIR / "embeddings" / "gene_to_embedding.pkl"
with open(EMB_PATH, "rb") as f:
    gene_to_embedding = pickle.load(f)

embed_dim = next(iter(gene_to_embedding.values())).shape[0]
print(f"Gene embeddings: {len(gene_to_embedding):,} genes x {embed_dim}-dim")

In [ ]:
BIOMART_FILE = DATA_DIR / "raw" / "mart_export.txt"
df_biomart = pd.read_csv(BIOMART_FILE)
df_pc = df_biomart[df_biomart["Gene type"] == "protein_coding"].dropna(
    subset=["Gene stable ID", "Gene name"]
)
gene_list = df_pc["Gene stable ID"].unique().tolist()

ensembl_to_symbol = (
    df_pc.drop_duplicates("Gene stable ID")
    .set_index("Gene stable ID")["Gene name"]
    .to_dict()
)
print(f"Protein-coding Ensembl IDs: {len(gene_list):,}")

In [ ]:
experiment = soma.open(SOMA_URI, mode="r")

obs_df = (
    experiment.obs.read(
        value_filter='assay == "10x 3\' v3" and is_primary_data == True',
        column_names=["soma_joinid", "cell_type_ontology_term_id", "cell_type"],
    )
    .concat()
    .to_pandas()
)
print(f"Total 10x v3 primary cells: {len(obs_df):,}")

obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(all_cell_values)].copy()
print(f"Blood cells in hierarchy: {len(obs_df):,}")

type_counts = obs_df["cell_type_ontology_term_id"].value_counts()
keep_types = type_counts[type_counts >= MIN_CELL_COUNT].index
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(keep_types)].copy()
print(
    f"After dropping < {MIN_CELL_COUNT} cells: {len(obs_df):,} cells, "
    f"{obs_df['cell_type_ontology_term_id'].nunique()} types"
)

cl_to_name = (
    obs_df.drop_duplicates("cell_type_ontology_term_id")
    .set_index("cell_type_ontology_term_id")["cell_type"]
    .to_dict()
)

# Create dataset with same split as training (same SEED)
joinids = obs_df["soma_joinid"].values
var_value_filter = f"feature_id in {gene_list}"

with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(coords=(joinids,)),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    var_df = query.var(column_names=["feature_id", "feature_name"]).concat().to_pandas()

    ds = ExperimentDataset(
        query,
        layer_name="raw",
        obs_column_names=["cell_type_ontology_term_id"],
        batch_size=effective_batch_size,
        shuffle=True,
        seed=SEED,
    )

# Use the same 80/20 split as training to get the held-out val set
_, val_ds = ds.random_split(0.8, 0.2, seed=SEED)

from attr import evolve
val_ds = evolve(val_ds, shuffle=False)

val_loader = experiment_dataloader(val_ds, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

print(f"\nVal set: ~{val_ds.shape[0]:,} cells x {val_ds.shape[1]:,} genes")

# Build gene embedding index from var metadata
col_indices = []
gene_names_mapped = []

seen_symbols = set()
for pos, (_, row) in enumerate(var_df.iterrows()):
    ensembl_id = row["feature_id"]
    symbol = ensembl_to_symbol.get(ensembl_id, row["feature_name"])
    if symbol in gene_to_embedding and symbol not in seen_symbols:
        col_indices.append(pos)
        gene_names_mapped.append(symbol)
        seen_symbols.add(symbol)

col_indices = np.array(col_indices)

gene_embs_tensor = torch.stack(
    [torch.from_numpy(gene_to_embedding[g]) for g in gene_names_mapped]
).to(device)

print(f"Genes with embeddings: {len(col_indices):,}/{len(var_df):,} ({100*len(col_indices)/len(var_df):.1f}%)")

## 4. Load Model & Run Validation

In [ ]:
class ScipherModel(nn.Module):
    def __init__(self, embed_dim, num_leaves, gene_embs, hidden_dim=256):
        super().__init__()
        self.embedder = AttentionPooling(embed_dim=embed_dim, hidden_dim=hidden_dim)
        self.classifier = WideNN(input_dim=embed_dim, output_dim=num_leaves)
        self.register_buffer("gene_embs", gene_embs)

    def forward(self, expression, mask):
        embeddings = self.gene_embs.unsqueeze(0).expand(expression.size(0), -1, -1)
        cell_embedding, attn_weights = self.embedder(embeddings, expression, mask)
        logits = self.classifier(cell_embedding)
        return logits, cell_embedding, attn_weights

model = ScipherModel(
    embed_dim=embed_dim, num_leaves=len(leaf_values),
    gene_embs=gene_embs_tensor, hidden_dim=256,
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel across {num_gpus} GPUs")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Loaded ScipherModel: {n_params:,} trainable parameters")

In [ ]:
loss_fn = MarginalizationLoss(
    marginalization_df, parent_child_df, exclusion_df,
    leaf_values, internal_values, mapping_dict,
    leaf_weight=ckpt["config"].get("leaf_weight", 7.0), device=device,
)

model.eval()
all_preds, all_labels = [], []
val_cell_embeddings = []
val_cell_types = []
val_losses = {"total": [], "leaf": [], "parent": []}

with torch.no_grad():
    for X_batch, obs_batch in val_loader:
        X = torch.from_numpy(X_batch) if isinstance(X_batch, np.ndarray) else torch.from_numpy(X_batch.toarray())
        X = X.float()

        X_mapped = X[:, col_indices]
        mask = (X_mapped > 0).to(device)
        expr_sum = X_mapped.sum(dim=1, keepdim=True).clamp(min=1e-10)
        expression = (X_mapped / expr_sum).to(device)

        labels = obs_batch["cell_type_ontology_term_id"].values
        y_batch = torch.tensor(
            [mapping_dict[t] for t in labels], device=device, dtype=torch.long
        )

        logits, cell_emb, _ = model(expression, mask)

        total_loss, loss_leafs, loss_parents = loss_fn(logits, y_batch)
        val_losses["total"].append(total_loss.item())
        val_losses["leaf"].append(loss_leafs.item())
        val_losses["parent"].append(loss_parents.item())

        preds = torch.argmax(logits, dim=1)

        is_leaf = torch.tensor(
            [y.item() in leaf_idx_set for y in y_batch], device=device
        )
        if is_leaf.sum() > 0:
            all_preds.extend(preds[is_leaf].cpu().numpy())
            all_labels.extend(y_batch[is_leaf].cpu().numpy())

        val_cell_embeddings.append(cell_emb.cpu().numpy())
        val_cell_types.extend(
            cl_to_name.get(t, t) for t in labels
        )

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
val_embeddings = np.concatenate(val_cell_embeddings, axis=0)

print(f"Validation complete: {len(all_labels):,} leaf-labeled cells, {val_embeddings.shape[0]:,} total cells")

## 5. Validation Loss

In [ ]:
avg_val_total = np.mean(val_losses["total"])
avg_val_leaf = np.mean(val_losses["leaf"])
avg_val_parent = np.mean(val_losses["parent"])

print(f"=== Validation Loss (avg over {len(val_losses['total'])} batches) ===")
print(f"  Total:  {avg_val_total:.4f}")
print(f"  Leaf:   {avg_val_leaf:.4f}")
print(f"  Parent: {avg_val_parent:.4f}")

# Compare with final training loss
if loss_history:
    # Average over last N batches of training to compare fairly
    n_compare = len(val_losses["total"])
    train_tail_total = np.mean(loss_history["total"][-n_compare:])
    train_tail_leaf = np.mean(loss_history["leaf"][-n_compare:])
    train_tail_parent = np.mean(loss_history["parent"][-n_compare:])

    print(f"\n=== Train vs Val Comparison (last {n_compare} batches of training) ===")
    print(f"  {'':20s} {'Train':>10s} {'Val':>10s} {'Gap':>10s}")
    print(f"  {'Total loss':20s} {train_tail_total:10.4f} {avg_val_total:10.4f} {avg_val_total - train_tail_total:+10.4f}")
    print(f"  {'Leaf loss':20s} {train_tail_leaf:10.4f} {avg_val_leaf:10.4f} {avg_val_leaf - train_tail_leaf:+10.4f}")
    print(f"  {'Parent loss':20s} {train_tail_parent:10.4f} {avg_val_parent:10.4f} {avg_val_parent - train_tail_parent:+10.4f}")

## 6. Classification Metrics

In [ ]:
acc = (all_preds == all_labels).mean()
macro_f1 = f1_score(all_labels, all_preds, average="macro")
weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"=== Validation (leaf-labeled cells) ===")
print(f"  Leaf accuracy: {acc:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Weighted F1:   {weighted_f1:.4f}")
print(f"  N samples:     {len(all_labels):,}")

# Per-class accuracy
rows = []
for idx in sorted(set(all_labels)):
    cl_id = idx_to_cl[idx]
    name = cl_to_name.get(cl_id, cl_id)
    mask_idx = all_labels == idx
    n = mask_idx.sum()
    class_acc = (all_preds[mask_idx] == all_labels[mask_idx]).mean()
    rows.append({"cell_type": name, "cl_id": cl_id, "n": n, "accuracy": class_acc})

df_eval = pd.DataFrame(rows).sort_values("accuracy")
print(f"\nPer-class accuracy:")
print(df_eval.to_string(index=False, float_format="{:.3f}".format))

## 7. UMAP Visualization

In [ ]:
import scanpy as sc

sc.settings.set_figure_params(dpi=100, frameon=False)

adata_umap = sc.AnnData(
    X=np.zeros((len(val_cell_types), 1)),
    obs=pd.DataFrame({"cell_type": val_cell_types}),
)
adata_umap.obsm["X_attn"] = val_embeddings

sc.pp.neighbors(adata_umap, use_rep="X_attn")
sc.tl.umap(adata_umap)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sc.pl.umap(
    adata_umap, color="cell_type", ax=ax, show=False,
    title=f"Attention-Pooled Cell Embeddings (epoch {ckpt['epoch']})",
    legend_loc="on data", legend_fontsize=6, frameon=False,
)

plt.tight_layout()
plt.savefig(
    DATA_DIR / "reports" / "eval_embedding_umap.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Saved to data/reports/eval_embedding_umap.png")